# Thougths (Analisis de avances y resultados durante el trabajo)

Dentro de este archivo "Jupyter python" se almacenara todo el trabajo necesario de analsis y entendimiento de la problematica a trabajar dado que es un problema de **Series Temporales (Time series and NowCasting)**. Como objetivo principal se busca estimar la inflacion de los combustibles y la calefaccion en Italia usando datos del mercado energetico que se pueden observar antes de que se publiquen las cifras oficiales. 

**Objetivo:** Construir un sistema de **Nowcasting** que permita estimar la inflacion energetica italiana **antes de que Eurostat publique oficialmente el dato**.


# Cual realmente es la problematica 

El problema nace de un desface temporal:

- Los precios del mercado de petroleo, gasolina, diesel y EUR/USD se observan **Diariamente**
- EL HICP italiano de combustibles se publica **mensualmente** y con retraso
- Por tanto durante varias semanas se sabe que esta ocurriendo en los mercados, pero todavia no se sabe oficialmente cuanto cambiaron los precios pagaron los consumidores italianos

¿Podemos utilizar informacion de los mercados energeticos disponible hoy para estimar como se está comportando la inflacion de combustibles en italia antes de que Eurstat publique el dato?

**Problema adiccional** Si hoy fuera por ejemplo el 15 de agosto, solo conocemos aproximadamente la mitad de los precios de mercado de agosto.
Para poder estimar la inflacion de agosto se necesita de la informacion representativa de todo agosto.


## La arquitectura conceptual de proyecto esta compuesta de la siguiente manera:

### Part 1 Forecasting de mercados
- **Brent:** Petroleo Brent
- **WTI:** Petroleo WTI
- **Gasoline Spot:** Gasolina Refinada
- **Diesel Spot:** Diesel Refinado
- **Heating Oil:** 
- **EUR/USD:** Tipo de cambio

Con un resultado / output de **Precios observados** + **Precios estiamdos**

Buscando **generar una agregacion mensual** continuamos con la siguiente parte
### Part 2 Consumer Price modelling

- **HICP transport fuel italy**
- **HICP heating fuels italy**

Con resultado / output de **NOWCAST inflation**


La **Parte 1 no es el objetivo final**. Es una herramienta para resolver un problema de informacion de la parte 2


## 1. Part 1 Forecasting de mercados

Se trata de constuir un modelo predictivo para las seis series de mercado, **aunque permite seleccionar un subconjunto si el tiempo es limitado** las 6 series son las previamente mencionadas.

Cada sereia inicialmente debe tratarse como un problema de forecasting independiente

**Que es lo que realmente se esta buscando evaluar?**

- **¿Que estas intentando predecir?** ej. target = Precio de mañana o target = retorno porcentual del dia siguiente o target = precio medio del proximo mes

Libertad de eleccion entre el horizonte temporal **(diario o mensual)**

## 2. Contra que compara el modelo?
En series financieras/commodities, una baseline muy razonable seria $\hat{y}_{t+1} = y_t$.

Es decir "El precio de mañana será igual al de hoy"

por lo que seria comparar Naive con Linear regression por ejemplo

y descubrir que quizas el modelo sofisticado no consigue superar de manera consistente al naive


## 3. Part 2 Consume price modeling

Aqui ya no se esta realizando la pregunta de ¿Cuanto costará Brent mañana?

**Ahora nos preguntamos** ¿Que informacion de los mercados explica o predice los precios que paga el consumidor italiano?

**Los targets ahora son:**
- **hcip_transport_fuels_it**
- **hicp_heating_it**

### EL HICP no es directamente inflacion
Esto es algo que recalcal para destacar a continuacion esta el siguiente ejemplo

- Jan HICP = 120
- Feb HICP = 122

No significa que tenga una inflacion del 122%, es un indice, lo relevante es su cambio. Por lo que para este probelma de **NowCasting** se prestaria mayor atencion al MoM, porque se busca determinar que esta pasando en el mes actual

### Primer transformacion economica importante: USD -> EUR

Todos los comodities estan cotizados en USD, mientras que estamos intentando explicar precios italianos.


### Que se usaria para predeicir el HICP

- **Transport fuels**
    - **Brent:** Con mayor prioridad
    - **WTI**
    - **Gasoline Spot:** Con mayor prioridad
    - **Diesel Spot:** Con mayor prioridad
    - **EUR/USD:** Con mayor prioridad
Dando output de **HICP transport fuels**

### Heating fuels

Para el **hicp_heating_it** se buscaria tener algo mas parecido a **heating_oil_ny + us_diesel_spot + brent + EURUSD --> HICP heating italy**

###  Problema con el pass-through
por ejemplo si hoy sube el brent un 10%, el consumidor italiano probablemente **no ve + 10 % mañana**
Crude oil
    ↓
Refinery
    ↓
Refined product
    ↓
Transport / storage
    ↓
Distributor
    ↓
Taxes / excise duties
    ↓
VAT
    ↓
Consumer

Se menciona que explicitamente refining margins, distribution costs, ecxise duties y el VAT son factores que retrasan o distorsionana la transmision.

### El gran problema tecnico: Frecuencia diaria vs Frecuencia mensual

Los mercados son diarios en represetacion de:
Jan 01
Jan 02
Jan 03
...

Mientras que el HICP presenta
Jan
Feb
Mar

Por lo que se debe transformar los mercados


### Batalla frente a la informacion a usar

**Los HICP** contienen datos desde 1996-01 -> 2025-12 que es alrededor de 30 meses

Pero **EURUSD** comienza en 1999 y **Diesel comienza en 2006**

por lo que si se decir de usar todas las variables tendria un periodo vivo de uso desde 2006 -> 2025 dando una cantidad de 20 años que es algo poco de observabilidad y salen las dos posibles soluciones

- **Modelo A:** Desde 1999-2025 con las variables de Brent, Gasoline, Heating Oil, EURUSD **que cuenta con con mas observaciones**
- **Modelo B:** Desde 2006-2025 con las variables de Brent, Gasoline, Diesel, Heating Oil, EURUSD **que cuenta con menos oservaciones pero con mas informacion (diesel - importante)**

## Batalla frente a datos falatantes (NaN)

Existen y se presentan por la diferencia de los calendarios diferentes entre Estados Unidos y ECB (Banco Central Europeo) por lo que por temas de diferencias en festivos y dias de "no trabajo" puede que exista informacion inexistente en alguno de los dos calendarios por lo que hay una gran probabilidad de alta cantidad de datos NaN 